# Notebook overview: R8_9.5_CO_bonding_bridging_split-Copy1.ipynb

## What this notebook does
Applies the refined bonding/bridging split with fixed pooled pre-disaster thresholds. It uses pre-disaster similarity and structural thresholds, applies them consistently across observed and random/counterfactual graphs, and identifies refined bridging ties using local constraint and degree/strength filters.

## Inputs used
- Dyad similarity table: dyad_unique_pair_similarity_{revision}.csv
- Observed full graph pickles for Oct2021, Nov2021, Jan2022, Feb2022
- Random/counterfactual graph pickles
- Pre-disaster pooled threshold settings for similarity, local constraint, and degree/strength

## Outputs created
- Observed bonding, bridging, refined-bridging, and unclassified graph pickles
- Counterfactual split graph pickles by run
- Threshold-tagged output paths and summary diagnostics

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install seaborn

In [ ]:
pip install geopy

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from shapely.geometry import LineString, Point
import pickle
import random
from matplotlib.lines import Line2D
import re

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
print("Current working directory:", os.getcwd())
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
revision = "R14_1"
boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
cf_out_dir = os.path.join(graph_poi_dir, "counterfactual_post", revision)

boot_dir_w = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat", revision)
os.makedirs(boot_dir_w, exist_ok=True)

boot_dir_w_count = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat_count", revision)
os.makedirs(boot_dir_w_count, exist_ok=True)

os.makedirs(cf_out_dir, exist_ok=True)

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

In [ ]:
# ============================================================
# 1. SETTINGS / PATHS
# ============================================================
revision = "R14_1"
sim_col = "race_inc_sim_mean"
weight_type = "race_inc_sim_mean"
weight_attr = "weight"

SIM_QUANTILES = [0.50, 0.70, 0.85]
EBC_QUANTILES = [0.50,
                #0.70, 0.85
                ]
LC_QUANTILES = [0.25, 0.35, 0.50]

PRE_MONTHS = {"Oct2021", "Nov2021"}
POST_MONTHS = {"Jan2022", "Feb2022"}
TIE_ORDER = ["bonding", "unclassified", "bridging_refined"]

# ------------------------------------------------------------
# REQUIRED INPUT TABLES
# ------------------------------------------------------------
pairwise_out_path = os.path.join(graph_dir, f"dyad_unique_pair_similarity_{revision}.csv")
edge_metrics_path = os.path.join(graph_dir, f"edge_betweenness_local_constraint_full_graph_{revision}.csv")

pairwise_df = pd.read_csv(pairwise_out_path, low_memory=False)
edge_metrics_df = pd.read_csv(edge_metrics_path, low_memory=False)

pairwise_df["user_i"] = pairwise_df["user_i"].astype(str)
pairwise_df["user_j"] = pairwise_df["user_j"].astype(str)

for col in ["source", "target", "revision", "month", "phase"]:
    edge_metrics_df[col] = edge_metrics_df[col].astype(str)

edge_metrics_df[["source", "target"]] = pd.DataFrame(
    edge_metrics_df.apply(
        lambda r: sorted([str(r["source"]), str(r["target"])]), axis=1
    ).tolist(),
    index=edge_metrics_df.index
)

print("Loaded pairwise_df:", pairwise_df.shape)
print("Loaded edge_metrics_df:", edge_metrics_df.shape)

In [ ]:
# ============================================================
# QUICK CHECK: DISTRIBUTION OF ALL-TIE LOCAL CONSTRAINT
# with 50%, 70%, 85% quantile lines
# ============================================================
lc = pd.to_numeric(edge_metrics_df["local_constraint_mean"], errors="coerce")
lc = lc.replace([np.inf, -np.inf], np.nan).dropna()

q50 = lc.quantile(0.50)
q70 = lc.quantile(0.70)
q85 = lc.quantile(0.85)

print("Local constraint")
print(lc.describe())
print("50% quantile:", q50)
print("70% quantile:", q70)
print("85% quantile:", q85)

plt.figure(figsize=(6, 4))

plt.hist(lc, bins=60, edgecolor="black", alpha=0.60)

plt.axvline(q50, linestyle="--", linewidth=2, label=f"50% = {q50:.3f}")
plt.axvline(q70, linestyle="--", linewidth=2, label=f"70% = {q70:.3f}")
plt.axvline(q85, linestyle="--", linewidth=2, label=f"85% = {q85:.3f}")

#plt.xlabel("Local constraint 30m", fontsize=14)
#plt.ylabel("Count", fontsize=14)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(loc="upper right", fontsize=20, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2. OBSERVED FULL GRAPHS
# ============================================================
graph_files = {
    "Oct2021": os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"),
    "Nov2021": os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl"),
    "Jan2022": os.path.join(graph_dir, f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
    "Feb2022": os.path.join(graph_dir, f"Feb2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
}

for k, v in graph_files.items():
    print(k, os.path.exists(v), v)

In [ ]:
# ============================================================
# 3. RANDOM BASE GRAPH FILES
# IMPORTANT:
# This assumes full random graphs exist as:
#   CF_graph_run0001_Oct2021_R10.pkl
# If your filenames differ, only edit list_random_base_graphs()
# and build_random_base_graph_path()
# ============================================================
def extract_run_number(filename):
    m = re.search(r"run(\d+)", filename)
    if m is None:
        raise ValueError(f"Could not parse run number from filename: {filename}")
    return int(m.group(1))


def list_random_base_graphs(boot_dir_w_count, revision):
    files = sorted([
        f for f in os.listdir(boot_dir_w_count)
        if f.startswith("CF_graph_run")
        and f.endswith(".pkl")
        and f"_Oct2021_{revision}.pkl" in f
        and "_bonding" not in f
        and "_bridging" not in f
        and "_bridging_refined" not in f
        and "_unclassified" not in f
    ])
    return files


def build_random_base_graph_path(boot_dir_w_count, revision, run):
    return os.path.join(
        boot_dir_w_count,
        f"CF_graph_run{run:04d}_Oct2021_{revision}.pkl"
    )


random_base_files = list_random_base_graphs(boot_dir_w_count, revision)
print("Random base graphs found:", len(random_base_files))
print(random_base_files[:5])

In [ ]:
# ============================================================
# 4. BASIC HELPERS
# ============================================================
def load_nx_graph(pkl_path):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    if not isinstance(G, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError(f"Loaded object is not a NetworkX graph: {type(G)} from {pkl_path}")
    return G


def save_nx_graph(G, pkl_path):
    os.makedirs(os.path.dirname(pkl_path), exist_ok=True)
    with open(pkl_path, "wb") as f:
        pickle.dump(G, f)


def ensure_str_nodes(G):
    if any(not isinstance(n, str) for n in G.nodes()):
        return nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    return G


def canonical_edge(u, v):
    u, v = str(u), str(v)
    return tuple(sorted((u, v)))


def graph_to_edge_df(G):
    rows = []
    for u, v, data in G.edges(data=True):
        source, target = canonical_edge(u, v)
        row = {"source": source, "target": target}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)


def get_phase(month):
    return "pre" if month in ["Oct2021", "Nov2021"] else "post"


def get_active_subgraph(G):
    active_nodes = [n for n, d in G.degree() if d > 0]
    return G.subgraph(active_nodes).copy()


def get_active_node_count(H):
    return sum(d > 0 for _, d in H.degree())


def qtag(q):
    return f"q{int(round(q * 100))}"


def combo_tag(sim_q, ebc_q, lc_q):
    return f"sim_{qtag(sim_q)}_ebc_{qtag(ebc_q)}_lc_{qtag(lc_q)}"

In [ ]:
# ============================================================
# 5. SPLIT INTO BONDING / BRIDGING
# FIXED SIMILARITY THRESHOLD FROM POOLED PRE-DISASTER ONLY
# ============================================================
def build_tie_lookup(pairwise_df, sim_col, quantile_used):
    pairwise_classified = pairwise_df.copy()
    pairwise_classified["user_i"] = pairwise_classified["user_i"].astype(str)
    pairwise_classified["user_j"] = pairwise_classified["user_j"].astype(str)

    # --------------------------------------------
    # pooled pre subset for threshold estimation
    # --------------------------------------------
    if {"month", "phase"}.issubset(pairwise_classified.columns):
        pre_mask = (
            pairwise_classified["month"].astype(str).isin(["Oct2021", "Nov2021"]) |
            pairwise_classified["phase"].astype(str).eq("pre")
        )
        pairwise_pre = pairwise_classified.loc[pre_mask].copy()
    else:
        # fallback if pairwise_df has no month/phase columns
        # then the table is already assumed to be the master dyad table
        pairwise_pre = pairwise_classified.copy()

    thr = pairwise_pre[sim_col].quantile(quantile_used)
    thr_tag = qtag(quantile_used)

    pairwise_classified["tie_type_proxy"] = np.where(
        pairwise_classified[sim_col] >= thr,
        "bonding_like",
        "bridging_like"
    )

    tie_lookup = dict(
        zip(
            zip(pairwise_classified["user_i"], pairwise_classified["user_j"]),
            pairwise_classified["tie_type_proxy"]
        )
    )

    print(f"\nPooled PRE similarity threshold ({thr_tag}) = {thr}")
    print(pairwise_classified["tie_type_proxy"].value_counts(dropna=False))

    return tie_lookup, thr, thr_tag


def split_graph_by_tie_lookup(G, tie_lookup):
    """
    Split graph edges into bonding-like and bridging-like subgraphs.
    Any edge not found in tie_lookup is assigned to bonding by default.
    """
    G_bonding = nx.Graph()
    G_bridging = nx.Graph()

    G_bonding.graph.update(G.graph)
    G_bridging.graph.update(G.graph)

    G_bonding.add_nodes_from(G.nodes(data=True))
    G_bridging.add_nodes_from(G.nodes(data=True))

    assigned_to_default_bonding = 0

    for u, v, data in G.edges(data=True):
        a, b = sorted([str(u), str(v)])
        tie_type = tie_lookup.get((a, b), None)

        if tie_type == "bridging_like":
            G_bridging.add_edge(u, v, **data)
        else:
            G_bonding.add_edge(u, v, **data)
            if tie_type is None:
                assigned_to_default_bonding += 1

    return G_bonding, G_bridging, assigned_to_default_bonding

In [ ]:
# ============================================================
# 6. FIXED FULL-GRAPH REFINEMENT THRESHOLDS
# USE ONE POOLED PRE-DISASTER BASELINE FOR ALL MONTHS / PHASES
# ============================================================
def compute_month_thresholds(edge_metrics_df, ebc_quantile_used, lc_quantile_used):
    edge_metrics_df = edge_metrics_df.copy()

    # --------------------------------------------
    # pooled pre-disaster full-graph edge metrics
    # --------------------------------------------
    pre_mask = (
        edge_metrics_df["month"].astype(str).isin(["Oct2021", "Nov2021"]) |
        edge_metrics_df["phase"].astype(str).eq("pre")
    )
    pre_df = edge_metrics_df.loc[pre_mask].copy()

    if pre_df.empty:
        raise ValueError("No pooled pre-disaster rows found in edge_metrics_df.")

    # one fixed baseline threshold per revision
    baseline_df = (
        pre_df
        .groupby(["revision"], as_index=False)
        .agg(
            edge_betweenness_threshold=("edge_betweenness", lambda x: x.quantile(ebc_quantile_used)),
            local_constraint_threshold=("local_constraint_mean", lambda x: x.quantile(lc_quantile_used))
        )
    )

    # copy that same baseline threshold to every month/phase row
    month_phase_df = (
        edge_metrics_df[["revision", "month", "phase"]]
        .drop_duplicates()
        .copy()
    )

    thresholds_df = month_phase_df.merge(
        baseline_df,
        on="revision",
        how="left"
    )

    thresholds_df["ebc_quantile_used"] = ebc_quantile_used
    thresholds_df["lc_quantile_used"] = lc_quantile_used

    print("\nUsing FIXED pooled PRE thresholds for all graphs:")
    print(baseline_df)

    return thresholds_df


def filter_by_edge_betweenness(bridge_edge_df, edge_metrics_df, thresholds_df):
    out = bridge_edge_df.merge(
        edge_metrics_df[
            ["source", "target", "revision", "month", "phase", "edge_betweenness"]
        ].drop_duplicates(),
        on=["source", "target", "revision", "month", "phase"],
        how="left"
    )

    out = out.merge(
        thresholds_df[
            ["revision", "month", "phase", "edge_betweenness_threshold"]
        ],
        on=["revision", "month", "phase"],
        how="left"
    )

    out["keep_ebc"] = out["edge_betweenness"] >= out["edge_betweenness_threshold"]
    out = out[out["keep_ebc"]].copy()
    return out


def filter_by_local_constraint(edge_df_after_ebc, edge_metrics_df, thresholds_df):
    out = edge_df_after_ebc.merge(
        edge_metrics_df[
            ["source", "target", "revision", "month", "phase", "local_constraint_mean"]
        ].drop_duplicates(),
        on=["source", "target", "revision", "month", "phase"],
        how="left"
    )

    out = out.merge(
        thresholds_df[
            ["revision", "month", "phase", "local_constraint_threshold"]
        ],
        on=["revision", "month", "phase"],
        how="left"
    )

    out["keep_local_constraint"] = out["local_constraint_mean"] <= out["local_constraint_threshold"]
    out = out[out["keep_local_constraint"]].copy()
    return out

In [ ]:
# ============================================================
# 7. GRAPH REFINEMENT / SUBTRACTION
# ============================================================
def keep_only_selected_edges(G, kept_edge_df):
    keep_edge_set = set(
        kept_edge_df.apply(lambda r: canonical_edge(r["source"], r["target"]), axis=1)
    )

    G_new = G.copy()
    edges_to_remove = []

    for u, v in G_new.edges():
        if canonical_edge(u, v) not in keep_edge_set:
            edges_to_remove.append((u, v))

    G_new.remove_edges_from(edges_to_remove)
    return G_new


def subtract_graph_edges(G_main, G_sub):
    """
    Return graph with edges in G_sub removed from G_main.
    Nodes are preserved; isolates are removed at the end.
    """
    G_out = G_main.copy()

    sub_edge_set = set(canonical_edge(u, v) for u, v in G_sub.edges())

    edges_to_remove = []
    for u, v in G_out.edges():
        if canonical_edge(u, v) in sub_edge_set:
            edges_to_remove.append((u, v))

    G_out.remove_edges_from(edges_to_remove)

    isolates = list(nx.isolates(G_out))
    G_out.remove_nodes_from(isolates)

    return G_out

In [ ]:
# ============================================================
# 8. SAVED PATH BUILDERS WITH THRESHOLD TAGS
# ============================================================
def build_observed_output_paths(graph_dir, revision, weight_type, month, sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    suffix_tag = f"{sim_thr_tag}_ebc_{ebc_thr_tag}_lc_{lc_thr_tag}"

    if month == "Oct2021":
        root = f"PDM_Oct2021_graph_POI_weighted_{revision}"
    elif month == "Nov2021":
        root = f"PDM_Nov2021_graph_POI_weighted_{revision}"
    elif month == "Jan2022":
        root = f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}"
    elif month == "Feb2022":
        root = f"Feb2022_remaining_pre_users_graph_POI_weighted_{revision}"
    else:
        raise ValueError(f"Unexpected month: {month}")

    return {
        "bonding": os.path.join(graph_dir, f"{root}_bonding-{weight_type}-{suffix_tag}.pkl"),
        "bridging": os.path.join(graph_dir, f"{root}_bridging-{weight_type}-{suffix_tag}.pkl"),
        "bridging_refined": os.path.join(graph_dir, f"{root}_bridging_refined-{weight_type}-{suffix_tag}.pkl"),
        "unclassified": os.path.join(graph_dir, f"{root}_unclassified-{weight_type}-{suffix_tag}.pkl"),
    }


def build_random_output_paths(boot_dir_w_count, revision, run, sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    suffix_tag = f"{sim_thr_tag}_ebc_{ebc_thr_tag}_lc_{lc_thr_tag}"
    root = f"CF_graph_run{run:04d}_Oct2021_{revision}"

    return {
        "bonding": os.path.join(boot_dir_w_count, f"{root}_bonding-{suffix_tag}.pkl"),
        "bridging": os.path.join(boot_dir_w_count, f"{root}_bridging-{suffix_tag}.pkl"),
        "bridging_refined": os.path.join(boot_dir_w_count, f"{root}_bridging_refined-{suffix_tag}.pkl"),
        "unclassified": os.path.join(boot_dir_w_count, f"{root}_unclassified-{suffix_tag}.pkl"),
    }

In [ ]:
# ============================================================
# 9. SPLIT + REFINE OBSERVED
# ============================================================
def run_observed_split_and_refine(
    graph_files,
    tie_lookup,
    thresholds_df,
    revision,
    weight_type,
    sim_thr_tag,
    ebc_thr_tag,
    lc_thr_tag
):
    split_rows = []
    refine_rows = []

    for month, graph_path in graph_files.items():
        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        print(f"\n📦 Observed | {month}")
        phase = get_phase(month)

        G = ensure_str_nodes(load_nx_graph(graph_path))

        G_bonding, G_bridging, assigned_to_default_bonding = split_graph_by_tie_lookup(G, tie_lookup)

        out_paths = build_observed_output_paths(
            graph_dir=graph_dir,
            revision=revision,
            weight_type=weight_type,
            month=month,
            sim_thr_tag=sim_thr_tag,
            ebc_thr_tag=ebc_thr_tag,
            lc_thr_tag=lc_thr_tag
        )

        save_nx_graph(G_bonding, out_paths["bonding"])
        save_nx_graph(G_bridging, out_paths["bridging"])

        print(f"   Original edges   : {G.number_of_edges():,}")
        print(f"   Bonding edges    : {G_bonding.number_of_edges():,}")
        print(f"   Bridging edges   : {G_bridging.number_of_edges():,}")

        split_rows.append({
            "revision": revision,
            "month": month,
            "phase": phase,
            "sim_thr_tag": sim_thr_tag,
            "ebc_thr_tag": ebc_thr_tag,
            "lc_thr_tag": lc_thr_tag,
            "original_edges": G.number_of_edges(),
            "bonding_edges": G_bonding.number_of_edges(),
            "bridging_edges": G_bridging.number_of_edges(),
            "assigned_to_default_bonding": assigned_to_default_bonding,
            "bonding_path": out_paths["bonding"],
            "bridging_path": out_paths["bridging"]
        })

        bridge_edge_df = graph_to_edge_df(G_bridging)
        bridge_edge_df["revision"] = revision
        bridge_edge_df["month"] = month
        bridge_edge_df["phase"] = phase
        
        # edge_df_after_ebc = filter_by_edge_betweenness(
        #     bridge_edge_df, edge_metrics_df, thresholds_df
        # )
        # "after_ebc_edges": len(edge_df_after_ebc),

        refined_edge_df = filter_by_local_constraint(
            bridge_edge_df,
            edge_metrics_df,
            thresholds_df
        )

        G_refined = keep_only_selected_edges(G_bridging, refined_edge_df)

        if G_refined.number_of_nodes() > 0:
            G_refined = nx.k_core(G_refined, k=3)

        G_unclassified = subtract_graph_edges(G_bridging, G_refined)

        save_nx_graph(G_refined, out_paths["bridging_refined"])
        save_nx_graph(G_unclassified, out_paths["unclassified"])

        print(f"   Bridging refined : {G_refined.number_of_edges():,}")
        print(f"   Unclassified     : {G_unclassified.number_of_edges():,}")

        refine_rows.append({
            "revision": revision,
            "month": month,
            "phase": phase,
            "sim_thr_tag": sim_thr_tag,
            "ebc_thr_tag": ebc_thr_tag,
            "lc_thr_tag": lc_thr_tag,
            "bridging_nodes": G_bridging.number_of_nodes(),
            "bridging_edges": G_bridging.number_of_edges(),
            "after_constraint_edges": len(refined_edge_df),
            "bridging_refined_nodes": G_refined.number_of_nodes(),
            "bridging_refined_edges": G_refined.number_of_edges(),
            "unclassified_nodes": G_unclassified.number_of_nodes(),
            "unclassified_edges": G_unclassified.number_of_edges(),
            "bridging_refined_path": out_paths["bridging_refined"],
            "unclassified_path": out_paths["unclassified"]
        })

    return pd.DataFrame(split_rows), pd.DataFrame(refine_rows)

In [ ]:
# ============================================================
# 10. SPLIT + REFINE RANDOM COUNTERFACTUAL
# IMPORTANT:
# refinement thresholds still come from observed full-graph table
# here we use Oct2021/pre thresholds for the chosen refine quantile
# ============================================================
def run_random_split_and_refine(
    boot_dir_w_count,
    tie_lookup,
    thresholds_df,
    revision,
    sim_thr_tag,
    ebc_thr_tag,
    lc_thr_tag
):
    split_rows = []
    refine_rows = []

    random_base_files = list_random_base_graphs(boot_dir_w_count, revision)

    if len(random_base_files) == 0:
        print("❌ No random base graphs found.")
        return pd.DataFrame(), pd.DataFrame()

    fixed_thr_df = thresholds_df[
        (thresholds_df["revision"] == str(revision)) &
        (thresholds_df["month"] == "Oct2021") &
        (thresholds_df["phase"] == "pre")
    ].copy()

    if fixed_thr_df.empty:
        raise ValueError("No Oct2021 pre threshold row found in thresholds_df.")

    for f in random_base_files:
        run = extract_run_number(f)
        graph_path = os.path.join(boot_dir_w_count, f)

        G = ensure_str_nodes(load_nx_graph(graph_path))

        G_bonding, G_bridging, assigned_to_default_bonding = split_graph_by_tie_lookup(G, tie_lookup)

        out_paths = build_random_output_paths(
            boot_dir_w_count=boot_dir_w_count,
            revision=revision,
            run=run,
            sim_thr_tag=sim_thr_tag,
            ebc_thr_tag=ebc_thr_tag,
            lc_thr_tag=lc_thr_tag
        )

        save_nx_graph(G_bonding, out_paths["bonding"])
        save_nx_graph(G_bridging, out_paths["bridging"])

        split_rows.append({
            "revision": revision,
            "run": run,
            "month": "Oct2021_counterfactual",
            "phase": "counterfactual",
            "sim_thr_tag": sim_thr_tag,
            "ebc_thr_tag": ebc_thr_tag,
            "lc_thr_tag": lc_thr_tag,
            #"refine_thr_tag": refine_thr_tag,
            "original_edges": G.number_of_edges(),
            "bonding_edges": G_bonding.number_of_edges(),
            "bridging_edges": G_bridging.number_of_edges(),
            "assigned_to_default_bonding": assigned_to_default_bonding,
            "bonding_path": out_paths["bonding"],
            "bridging_path": out_paths["bridging"]
        })

        bridge_edge_df = graph_to_edge_df(G_bridging)
        bridge_edge_df["revision"] = revision
        bridge_edge_df["month"] = "Oct2021"
        bridge_edge_df["phase"] = "pre"

        # edge_df_after_ebc = filter_by_edge_betweenness(
        #     bridge_edge_df, edge_metrics_df, fixed_thr_df
        # )

        refined_edge_df = filter_by_local_constraint(
            #edge_df_after_ebc, 
            bridge_edge_df, 
            edge_metrics_df, fixed_thr_df
        )

        G_refined = keep_only_selected_edges(G_bridging, refined_edge_df)

        if G_refined.number_of_nodes() > 0:
            G_refined = nx.k_core(G_refined, k=3)

        G_unclassified = subtract_graph_edges(G_bridging, G_refined)

        save_nx_graph(G_refined, out_paths["bridging_refined"])
        save_nx_graph(G_unclassified, out_paths["unclassified"])

        refine_rows.append({
            "revision": revision,
            "run": run,
            "month": "Oct2021_counterfactual",
            "phase": "counterfactual",
            "sim_thr_tag": sim_thr_tag,
            #"refine_thr_tag": refine_thr_tag,
            "ebc_thr_tag": ebc_thr_tag,
            "lc_thr_tag": lc_thr_tag,
            "bridging_nodes": G_bridging.number_of_nodes(),
            "bridging_edges": G_bridging.number_of_edges(),
            "after_constraint_edges": len(refined_edge_df),
            #"after_ebc_edges": len(edge_df_after_ebc),
            "bridging_refined_nodes": G_refined.number_of_nodes(),
            "bridging_refined_edges": G_refined.number_of_edges(),
            "unclassified_nodes": G_unclassified.number_of_nodes(),
            "unclassified_edges": G_unclassified.number_of_edges(),
            "bridging_refined_path": out_paths["bridging_refined"],
            "unclassified_path": out_paths["unclassified"]
        })

    return pd.DataFrame(split_rows), pd.DataFrame(refine_rows)

In [ ]:
# ============================================================
# 11. NETWORK METRICS
# ============================================================
def summarize_network(H, weight_attr="weight", centrality_denominator_n=None):
    Hm = get_active_subgraph(H)

    N = Hm.number_of_nodes()
    E = Hm.number_of_edges()

    if N == 0:
        return {
            "nodes": 0,
            "edges": 0,
            "mean_strength": 0.0,
            "mean_degree": 0.0,
            "mean_weighted_degree_centrality": 0.0,
            "median_weighted_degree_centrality": 0.0,
            "heterogeneity": 0.0,
            "density": 0.0,
            "avg_clustering_unweighted": 0.0,
            "avg_clustering_weighted": 0.0,
            "largest_cc_size": 0,
            "largest_cc_share": 0.0,
            "centrality_denominator_n": centrality_denominator_n if centrality_denominator_n is not None else 0
        }

    degs = np.array([d for _, d in Hm.degree()], dtype=float)
    degs_w = np.array([d for _, d in Hm.degree(weight=weight_attr)], dtype=float)

    mean_degree = float(degs.mean()) if degs.size else 0.0
    mean_strength = float(degs_w.mean()) if degs_w.size else 0.0

    denom_n = centrality_denominator_n if centrality_denominator_n is not None else N
    denom = denom_n - 1

    if denom > 0:
        weighted_degree_centrality = degs_w / denom
    else:
        weighted_degree_centrality = np.zeros_like(degs_w, dtype=float)

    mean_weighted_degree_centrality = float(weighted_degree_centrality.mean()) if weighted_degree_centrality.size else 0.0
    median_weighted_degree_centrality = float(np.median(weighted_degree_centrality)) if weighted_degree_centrality.size else 0.0

    hetero = float(degs_w.std() / mean_strength) if mean_strength > 0 else 0.0
    raw_density = nx.density(Hm)

    if E > 0:
        largest_cc = max(nx.connected_components(Hm), key=len)
        largest_cc_size = len(largest_cc)
        largest_cc_share = largest_cc_size / N
    else:
        largest_cc_size = 0
        largest_cc_share = 0.0

    return {
        "nodes": N,
        "edges": E,
        "mean_strength": mean_strength,
        "mean_degree": mean_degree,
        "mean_weighted_degree_centrality": mean_weighted_degree_centrality,
        "median_weighted_degree_centrality": median_weighted_degree_centrality,
        "heterogeneity": hetero,
        "density": raw_density,
        "avg_clustering_unweighted": float(nx.average_clustering(Hm)),
        "avg_clustering_weighted": float(nx.average_clustering(Hm, weight=weight_attr)),
        "largest_cc_size": largest_cc_size,
        "largest_cc_share": largest_cc_share,
        "centrality_denominator_n": denom_n
    }


def compute_pre_reference_denominators(sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    ref_paths = build_observed_output_paths(
        graph_dir=graph_dir,
        revision=revision,
        weight_type=weight_type,
        month="Oct2021",
        sim_thr_tag=sim_thr_tag,
        ebc_thr_tag=ebc_thr_tag,
        lc_thr_tag=lc_thr_tag
    )

    ref_denoms = {}
    for tie_type in TIE_ORDER:
        path = ref_paths[tie_type]
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing pre reference graph for {tie_type}: {path}")
        G_ref = ensure_str_nodes(load_nx_graph(path))
        ref_denoms[tie_type] = get_active_node_count(G_ref)

    return ref_denoms

In [ ]:
# ============================================================
# 12. CENTRALITIES
# ============================================================
def compute_node_centralities_contact_fast(G, weight="weight", approx_betweenness_k=500, seed=0):
    if G.number_of_nodes() == 0:
        return pd.DataFrame(
            columns=[
                "raw_degree", "strength", "closeness_centrality",
                "clustering_coefficient", "betweenness_centrality"
            ]
        )

    inv = {}
    for u, v, d in G.edges(data=True):
        w = d.get(weight, 0.0)
        inv[(u, v)] = (1.0 / w) if (w is not None and w > 0) else np.inf

    nx.set_edge_attributes(G, inv, "inv_weight")

    out = pd.DataFrame(index=list(G.nodes()))
    out["raw_degree"] = pd.Series(dict(G.degree()))
    out["strength"] = pd.Series(dict(G.degree(weight=weight)))
    out["closeness_centrality"] = pd.Series(
        nx.closeness_centrality(G, distance="inv_weight")
    )
    out["clustering_coefficient"] = pd.Series(nx.clustering(G, weight=None))

    if approx_betweenness_k is None or G.number_of_nodes() <= approx_betweenness_k:
        out["betweenness_centrality"] = pd.Series(
            nx.betweenness_centrality(G, weight="inv_weight")
        )
    else:
        out["betweenness_centrality"] = pd.Series(
            nx.betweenness_centrality(G, k=approx_betweenness_k, seed=seed, weight="inv_weight")
        )

    return out

In [ ]:
# ============================================================
# 13. BUILD FILE PATHS FOR METRICS / CENTRALITIES
# ============================================================
def build_split_graph_files_for_revision_combo(sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    out = {}
    for month in ["Oct2021", "Nov2021", "Jan2022", "Feb2022"]:
        p = build_observed_output_paths(
            graph_dir=graph_dir,
            revision=revision,
            weight_type=weight_type,
            month=month,
            sim_thr_tag=sim_thr_tag,
            ebc_thr_tag=ebc_thr_tag,
            lc_thr_tag=lc_thr_tag
        )
        out[f"{month}_bonding"] = p["bonding"]
        out[f"{month}_unclassified"] = p["unclassified"]
        out[f"{month}_bridging_refined"] = p["bridging_refined"]
    return out


def parse_graph_name(graph_name):
    parts = graph_name.split("_")
    month = parts[0]
    tie_type = "_".join(parts[1:])
    phase = "pre" if month in PRE_MONTHS else "post"
    return month, tie_type, phase


def list_random_runs_for_combo(boot_dir_w_count, revision, sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    suffix_tag = f"{sim_thr_tag}_ebc_{ebc_thr_tag}_lc_{lc_thr_tag}"

    files = os.listdir(boot_dir_w_count)
    runs = sorted({
        extract_run_number(f)
        for f in files
        if f.startswith("CF_graph_run")
        and f.endswith(".pkl")
        and f"_Oct2021_{revision}_" in f
        and f"-{suffix_tag}.pkl" in f
        and (
            "_bonding-" in f
            or "_unclassified-" in f
            or "_bridging_refined-" in f
        )
    })
    return runs


def build_random_split_graph_paths(boot_dir_w_count, revision, run, sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    p = build_random_output_paths(
        boot_dir_w_count=boot_dir_w_count,
        revision=revision,
        run=run,
        sim_thr_tag=sim_thr_tag,
        ebc_thr_tag=ebc_thr_tag,
        lc_thr_tag=lc_thr_tag
    )

    return {
        "bonding": p["bonding"],
        "unclassified": p["unclassified"],
        "bridging_refined": p["bridging_refined"],
    }

In [ ]:
# ============================================================
# 14. COMPUTE METRICS + CENTRALITIES FOR ONE THRESHOLD COMBO
# ============================================================
def run_metrics_and_centralities_for_combo(sim_thr_tag, ebc_thr_tag, lc_thr_tag):
    combo = f"sim_{sim_thr_tag}_ebc_{ebc_thr_tag}_lc_{lc_thr_tag}"
    ref_denoms = compute_pre_reference_denominators(sim_thr_tag, ebc_thr_tag, lc_thr_tag)

    observed_rows = []
    graph_files_combo = build_split_graph_files_for_revision_combo(sim_thr_tag, ebc_thr_tag, lc_thr_tag)

    for graph_name, graph_path in graph_files_combo.items():
        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        G = ensure_str_nodes(load_nx_graph(graph_path))
        month, tie_type, phase = parse_graph_name(graph_name)

        net = summarize_network(
            G,
            weight_attr=weight_attr,
            centrality_denominator_n=ref_denoms[tie_type]
        )

        net.update({
            "revision": revision,
            "month": month,
            "phase": phase,
            "tie_type": tie_type,
            "dataset": "observed",
            "run": np.nan,
            "sim_thr_tag": sim_thr_tag,
            "ebc_thr_tag": ebc_thr_tag,
            "lc_thr_tag": lc_thr_tag,
            "combo": combo
        })

        observed_rows.append(net)

    observed_df = pd.DataFrame(observed_rows)

    random_rows = []
    runs = list_random_runs_for_combo(boot_dir_w_count, revision, sim_thr_tag, ebc_thr_tag, lc_thr_tag)

    for run in runs:
        graph_files = build_random_split_graph_paths(
            boot_dir_w_count, revision, run, sim_thr_tag, ebc_thr_tag, lc_thr_tag
        )

        for tie_type, graph_path in graph_files.items():
            if not os.path.exists(graph_path):
                print(f"❌ Missing: {graph_path}")
                continue

            G = ensure_str_nodes(load_nx_graph(graph_path))

            net = summarize_network(
                G,
                weight_attr=weight_attr,
                centrality_denominator_n=ref_denoms[tie_type]
            )

            net.update({
                "revision": revision,
                "run": run,
                "month": "Oct2021_counterfactual",
                "phase": "counterfactual",
                "tie_type": tie_type,
                "dataset": "random",
                "sim_thr_tag": sim_thr_tag,
                "ebc_thr_tag": ebc_thr_tag,
                "lc_thr_tag": lc_thr_tag,
                "combo": combo
            })

            random_rows.append(net)

    random_df = pd.DataFrame(random_rows)

    node_centrality_rows = []

    for graph_name, graph_path in graph_files_combo.items():
        if not os.path.exists(graph_path):
            continue

        G = ensure_str_nodes(load_nx_graph(graph_path))
        Gm = get_active_subgraph(G)
        month, tie_type, phase = parse_graph_name(graph_name)

        node_cent = compute_node_centralities_contact_fast(
            Gm,
            weight="weight",
            approx_betweenness_k=500,
            seed=0
        )

        node_cent = node_cent.reset_index().rename(columns={"index": "node"})
        node_cent["month"] = month
        node_cent["phase"] = phase
        node_cent["tie_type"] = tie_type
        node_cent["revision"] = revision
        node_cent["run"] = np.nan
        node_cent["dataset"] = "observed"
        node_cent["sim_thr_tag"] = sim_thr_tag
        node_cent["ebc_thr_tag"] = ebc_thr_tag
        node_cent["lc_thr_tag"] = lc_thr_tag
        node_cent["combo"] = combo

        node_centrality_rows.append(node_cent)

    observed_node_df = pd.concat(node_centrality_rows, ignore_index=True) if len(node_centrality_rows) > 0 else pd.DataFrame()

    random_node_centrality_rows = []

    for run in runs:
        graph_files = build_random_split_graph_paths(
            boot_dir_w_count, revision, run, sim_thr_tag, ebc_thr_tag, lc_thr_tag
        )

        for tie_type, graph_path in graph_files.items():
            if not os.path.exists(graph_path):
                continue

            G = ensure_str_nodes(load_nx_graph(graph_path))
            Gm = get_active_subgraph(G)

            node_cent = compute_node_centralities_contact_fast(
                Gm,
                weight="weight",
                approx_betweenness_k=500,
                seed=0
            )

            node_cent = node_cent.reset_index().rename(columns={"index": "node"})
            node_cent["month"] = "Oct2021_counterfactual"
            node_cent["phase"] = "counterfactual"
            node_cent["tie_type"] = tie_type
            node_cent["revision"] = revision
            node_cent["run"] = run
            node_cent["dataset"] = "random"
            node_cent["sim_thr_tag"] = sim_thr_tag
            node_cent["ebc_thr_tag"] = ebc_thr_tag
            node_cent["lc_thr_tag"] = lc_thr_tag
            node_cent["combo"] = combo

            random_node_centrality_rows.append(node_cent)

    random_node_df = pd.concat(random_node_centrality_rows, ignore_index=True) if len(random_node_centrality_rows) > 0 else pd.DataFrame()

    network_metrics_all_df = pd.concat([observed_df, random_df], ignore_index=True)
    node_metrics_centrality_split_df = pd.concat([observed_node_df, random_node_df], ignore_index=True)

    return observed_df, random_df, network_metrics_all_df, observed_node_df, random_node_df, node_metrics_centrality_split_df

In [ ]:
# ============================================================
# 15. WEIGHTED DEGREE PLOTS FOR ONE THRESHOLD COMBO
# ============================================================
def compute_density_curve(strengths, bins):
    strengths = pd.to_numeric(strengths, errors="coerce")
    strengths = strengths[(strengths > 0) & (~pd.isna(strengths))]

    if len(strengths) == 0:
        return np.full(len(bins) - 1, np.nan), np.nan

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    counts = counts.astype(float)
    counts[counts == 0] = np.nan

    return counts, strengths.mean()


def average_month_curves(df_sub, months, bins):
    curves = []
    month_means = []

    for m in months:
        dfi = df_sub[df_sub["month"] == m].copy()
        curve, mean_val = compute_density_curve(dfi["strength"], bins)

        if np.all(np.isnan(curve)):
            continue

        curves.append(curve)
        month_means.append(mean_val)

    if len(curves) == 0:
        empty = np.full(len(bins) - 1, np.nan)
        return empty, np.nan

    curves = np.array(curves, dtype=float)
    valid_rows = ~np.all(np.isnan(curves), axis=1)
    curves = curves[valid_rows]

    if len(curves) == 0:
        empty = np.full(len(bins) - 1, np.nan)
        return empty, np.nan

    mean_curve = np.nanmean(curves, axis=0)
    mean_curve[mean_curve == 0] = np.nan

    mean_strength = np.nanmean(month_means) if len(month_means) > 0 else np.nan
    return mean_curve, mean_strength


def average_run_curves_with_band(df_sub, bins):
    curves = []
    run_means = []

    for run in sorted(df_sub["run"].dropna().unique()):
        dfi = df_sub[df_sub["run"] == run].copy()
        curve, mean_val = compute_density_curve(dfi["strength"], bins)

        if np.all(np.isnan(curve)):
            continue

        curves.append(curve)
        run_means.append(mean_val)

    if len(curves) == 0:
        empty = np.full(len(bins) - 1, np.nan)
        return empty, empty, empty, np.nan

    curves = np.array(curves, dtype=float)
    valid_rows = ~np.all(np.isnan(curves), axis=1)
    curves = curves[valid_rows]

    if len(curves) == 0:
        empty = np.full(len(bins) - 1, np.nan)
        return empty, empty, empty, np.nan

    mean_curve = np.nanmean(curves, axis=0)
    std_curve = np.nanstd(curves, axis=0)

    lower = mean_curve - std_curve
    upper = mean_curve + std_curve

    mean_curve[mean_curve <= 0] = np.nan
    lower[lower <= 0] = np.nan
    upper[upper <= 0] = np.nan

    mean_strength = np.nanmean(run_means) if len(run_means) > 0 else np.nan
    return mean_curve, lower, upper, mean_strength


def plot_weighted_degree_for_combo(df, combo_label):
    tie_types = ["bonding", "bridging_refined", "unclassified"]

    condition_colors = {
        "pre": "seagreen",
        "post": "orangered",
        "counterfactual": "darkred"
    }

    marker_map = {
        "pre": "s",
        "post": "o",
        "counterfactual": "v"
    }

    tie_label_map = {
        "bonding": "Bonding Ties",
        "bridging_refined": "Bridging Ties",
        "unclassified": "Unclassified"
    }

    pre_months = ["Oct2021", "Nov2021"]
    post_months = ["Jan2022", "Feb2022"]

    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 2.6),
        sharey=True
    )

    for ax, tie_type in zip(axes, tie_types):
        df_tie = df[(df["tie_type"] == tie_type) & (df["strength"] > 0)].copy()

        if df_tie.empty:
            ax.set_title(tie_label_map[tie_type], fontsize=16)
            ax.axis("off")
            continue

        bins = np.logspace(
            np.log10(df_tie["strength"].min()),
            np.log10(df_tie["strength"].max()),
            25
        )
        bin_centers = np.sqrt(bins[:-1] * bins[1:])

        df_pre = df[
            (df["dataset"] == "observed") &
            (df["phase"] == "pre") &
            (df["tie_type"] == tie_type)
        ].copy()

        pre_curve, pre_mean = average_month_curves(df_pre, pre_months, bins)

        ax.plot(
            bin_centers,
            pre_curve,
            color=condition_colors["pre"],
            marker=marker_map["pre"],
            markerfacecolor=condition_colors["pre"],
            markeredgecolor=condition_colors["pre"],
            linewidth=2,
            label="Pre"
        )

        if pd.notna(pre_mean) and pre_mean > 0:
            ax.axvline(
                pre_mean,
                color=condition_colors["pre"],
                linestyle="--",
                linewidth=1.3,
                alpha=0.9
            )

        df_post = df[
            (df["dataset"] == "observed") &
            (df["phase"] == "post") &
            (df["tie_type"] == tie_type)
        ].copy()

        post_curve, post_mean = average_month_curves(df_post, post_months, bins)

        ax.plot(
            bin_centers,
            post_curve,
            color=condition_colors["post"],
            marker=marker_map["post"],
            markerfacecolor=condition_colors["post"],
            markeredgecolor=condition_colors["post"],
            linewidth=2,
            linestyle="-.",
            label="Post"
        )

        if pd.notna(post_mean) and post_mean > 0:
            ax.axvline(
                post_mean,
                color=condition_colors["post"],
                linestyle=":",
                linewidth=1.3,
                alpha=0.9
            )

        df_rand = df[
            (df["dataset"] == "random") &
            (df["phase"] == "counterfactual") &
            (df["tie_type"] == tie_type)
        ].copy()

        rand_curve, rand_low, rand_high, rand_mean = average_run_curves_with_band(df_rand, bins)

        ax.plot(
            bin_centers,
            rand_curve,
            color=condition_colors["counterfactual"],
            marker=marker_map["counterfactual"],
            markerfacecolor=condition_colors["counterfactual"],
            markeredgecolor=condition_colors["counterfactual"],
            linewidth=2,
            linestyle="-",
            label="Counterfactual"
        )

        ax.fill_between(
            bin_centers,
            rand_low,
            rand_high,
            color=condition_colors["counterfactual"],
            alpha=0.20
        )

        if pd.notna(rand_mean) and rand_mean > 0:
            ax.axvline(
                rand_mean,
                color=condition_colors["counterfactual"],
                linestyle=(0, (3, 2)),
                linewidth=1.3,
                alpha=0.9
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=16)
        ax.set_title(tie_label_map[tie_type], fontsize=16)

    axes[0].set_ylabel("P(k)", fontsize=16)
    fig.suptitle(combo_label, fontsize=14)

    plt.subplots_adjust(wspace=0.22)
    plt.tight_layout(rect=[0, 0, 0.88, 0.95])
    plt.show()

In [ ]:
# ============================================================
# 16. MAIN ROBUSTNESS LOOP: RUN ALL COMBINATIONS
#     SKIP COMBOS IF FINAL OUTPUT FILES ALREADY EXIST
# ============================================================
all_split_observed = []
all_refine_observed = []
all_split_random = []
all_refine_random = []
all_network_metrics = []
all_node_centralities = []
all_combo_info = []

robustness_dir = os.path.join(graph_dir, f"robustness_threshold_grid_{revision}")
os.makedirs(robustness_dir, exist_ok=True)

for quantile_used in SIM_QUANTILES:
    tie_lookup, thr, thr_tag = build_tie_lookup(pairwise_df, sim_col, quantile_used)

    for ebc_quantile_used in EBC_QUANTILES:
        for lc_quantile_used in LC_QUANTILES:
            ebc_thr_tag = qtag(ebc_quantile_used)
            lc_thr_tag = qtag(lc_quantile_used)

            current_combo = combo_tag(quantile_used, ebc_quantile_used, lc_quantile_used)

            network_metrics_out_path = os.path.join(
                robustness_dir,
                f"network_metrics_bonding_bridging_refined_observed_plus_random_{revision}-{weight_type}-{current_combo}.csv"
            )
            node_metrics_out_path = os.path.join(
                robustness_dir,
                f"node_metrics_centrality_bonding_bridging_refined_observed_plus_random_{revision}-{weight_type}-{current_combo}.csv"
            )

            # ----------------------------------------------------
            # skip if final combo outputs already exist
            # ----------------------------------------------------
            if os.path.exists(network_metrics_out_path) and os.path.exists(node_metrics_out_path):
                print("\n" + "=" * 90)
                print(f"SKIPPING EXISTING: {current_combo}")
                print("=" * 90)
                print("✅ Found:", network_metrics_out_path)
                print("✅ Found:", node_metrics_out_path)

                all_combo_info.append({
                    "revision": revision,
                    "sim_quantile_used": quantile_used,
                    "sim_threshold": thr,
                    "sim_thr_tag": thr_tag,
                    "ebc_quantile_used": ebc_quantile_used,
                    "ebc_thr_tag": ebc_thr_tag,
                    "lc_quantile_used": lc_quantile_used,
                    "lc_thr_tag": lc_thr_tag,
                    "combo": current_combo,
                    "network_metrics_out_path": network_metrics_out_path,
                    "node_metrics_out_path": node_metrics_out_path
                })
                continue

            print("\n" + "=" * 90)
            print(f"RUNNING: {current_combo}")
            print("=" * 90)

            thresholds_df = compute_month_thresholds(
                edge_metrics_df,
                ebc_quantile_used,
                lc_quantile_used
            )

            summary_split_observed, summary_refine_observed = run_observed_split_and_refine(
                graph_files=graph_files,
                tie_lookup=tie_lookup,
                thresholds_df=thresholds_df,
                revision=revision,
                weight_type=weight_type,
                sim_thr_tag=thr_tag,
                ebc_thr_tag=ebc_thr_tag,
                lc_thr_tag=lc_thr_tag
            )

            summary_split_random, summary_refine_random = run_random_split_and_refine(
                boot_dir_w_count=boot_dir_w_count,
                tie_lookup=tie_lookup,
                thresholds_df=thresholds_df,
                revision=revision,
                sim_thr_tag=thr_tag,
                ebc_thr_tag=ebc_thr_tag,
                lc_thr_tag=lc_thr_tag
            )

            observed_df, random_df, network_metrics_all_df, observed_node_df, random_node_df, node_metrics_centrality_split_df = run_metrics_and_centralities_for_combo(
                sim_thr_tag=thr_tag,
                ebc_thr_tag=ebc_thr_tag,
                lc_thr_tag=lc_thr_tag
            )

            network_metrics_all_df.to_csv(network_metrics_out_path, index=False)
            node_metrics_centrality_split_df.to_csv(node_metrics_out_path, index=False)

            print("✅ Saved network metrics:", network_metrics_out_path)
            print("✅ Saved node centralities:", node_metrics_out_path)

            all_combo_info.append({
                "revision": revision,
                "sim_quantile_used": quantile_used,
                "sim_threshold": thr,
                "sim_thr_tag": thr_tag,
                "ebc_quantile_used": ebc_quantile_used,
                "ebc_thr_tag": ebc_thr_tag,
                "lc_quantile_used": lc_quantile_used,
                "lc_thr_tag": lc_thr_tag,
                "combo": current_combo,
                "network_metrics_out_path": network_metrics_out_path,
                "node_metrics_out_path": node_metrics_out_path
            })

            if len(summary_split_observed) > 0:
                summary_split_observed = summary_split_observed.copy()
                summary_split_observed["combo"] = current_combo
                all_split_observed.append(summary_split_observed)

            if len(summary_refine_observed) > 0:
                summary_refine_observed = summary_refine_observed.copy()
                summary_refine_observed["combo"] = current_combo
                all_refine_observed.append(summary_refine_observed)

            if len(summary_split_random) > 0:
                summary_split_random = summary_split_random.copy()
                summary_split_random["combo"] = current_combo
                all_split_random.append(summary_split_random)

            if len(summary_refine_random) > 0:
                summary_refine_random = summary_refine_random.copy()
                summary_refine_random["combo"] = current_combo
                all_refine_random.append(summary_refine_random)

            if len(network_metrics_all_df) > 0:
                all_network_metrics.append(network_metrics_all_df.copy())

            if len(node_metrics_centrality_split_df) > 0:
                all_node_centralities.append(node_metrics_centrality_split_df.copy())

# ------------------------------------------------------------
# combine master summary tables
# ------------------------------------------------------------
combo_info_df = pd.DataFrame(all_combo_info)

split_observed_all_df = pd.concat(all_split_observed, ignore_index=True) if len(all_split_observed) > 0 else pd.DataFrame()
refine_observed_all_df = pd.concat(all_refine_observed, ignore_index=True) if len(all_refine_observed) > 0 else pd.DataFrame()
split_random_all_df = pd.concat(all_split_random, ignore_index=True) if len(all_split_random) > 0 else pd.DataFrame()
refine_random_all_df = pd.concat(all_refine_random, ignore_index=True) if len(all_refine_random) > 0 else pd.DataFrame()
network_metrics_master_df = pd.concat(all_network_metrics, ignore_index=True) if len(all_network_metrics) > 0 else pd.DataFrame()
node_centralities_master_df = pd.concat(all_node_centralities, ignore_index=True) if len(all_node_centralities) > 0 else pd.DataFrame()

# ------------------------------------------------------------
# save master summary tables
# ------------------------------------------------------------
combo_info_path = os.path.join(robustness_dir, f"combo_info_{revision}.csv")
split_observed_path = os.path.join(robustness_dir, f"split_observed_summary_{revision}.csv")
refine_observed_path = os.path.join(robustness_dir, f"refine_observed_summary_{revision}.csv")
split_random_path = os.path.join(robustness_dir, f"split_random_summary_{revision}.csv")
refine_random_path = os.path.join(robustness_dir, f"refine_random_summary_{revision}.csv")
network_metrics_master_path = os.path.join(robustness_dir, f"network_metrics_all_combos_{revision}.csv")
node_centralities_master_path = os.path.join(robustness_dir, f"node_centralities_all_combos_{revision}.csv")

combo_info_df.to_csv(combo_info_path, index=False)
split_observed_all_df.to_csv(split_observed_path, index=False)
refine_observed_all_df.to_csv(refine_observed_path, index=False)
split_random_all_df.to_csv(split_random_path, index=False)
refine_random_all_df.to_csv(refine_random_path, index=False)
network_metrics_master_df.to_csv(network_metrics_master_path, index=False)
node_centralities_master_df.to_csv(node_centralities_master_path, index=False)

print("\n✅ Saved master summary files:")
print(combo_info_path)
print(split_observed_path)
print(refine_observed_path)
print(split_random_path)
print(refine_random_path)
print(network_metrics_master_path)
print(node_centralities_master_path)

In [ ]:
# ============================================================
# 17. REBUILD MASTER TABLES FROM ALREADY-SAVED COMBO FILES
#     then load and plot
# ============================================================
robustness_dir = os.path.join(graph_dir, f"robustness_threshold_grid_{revision}")

combo_info_path = os.path.join(robustness_dir, f"combo_info_{revision}.csv")
combo_info_df = pd.read_csv(combo_info_path)

print("Loaded combo_info_df:", combo_info_df.shape)
print("combo_info_df combos:")
print(sorted(combo_info_df["combo"].dropna().unique()))

network_dfs = []
node_dfs = []

for _, row in combo_info_df.iterrows():
    combo = row["combo"]
    net_path = row["network_metrics_out_path"]
    node_path = row["node_metrics_out_path"]

    if os.path.exists(net_path) and os.path.getsize(net_path) > 0:
        try:
            df_net = pd.read_csv(net_path)
            if not df_net.empty:
                network_dfs.append(df_net)
                print(f"✅ loaded network metrics for {combo}")
            else:
                print(f"⚠️ empty network metrics for {combo}")
        except Exception as e:
            print(f"❌ failed network metrics for {combo}: {e}")
    else:
        print(f"⚠️ missing network metrics file for {combo}")

    if os.path.exists(node_path) and os.path.getsize(node_path) > 0:
        try:
            df_node = pd.read_csv(node_path)
            if not df_node.empty:
                node_dfs.append(df_node)
                print(f"✅ loaded node centralities for {combo}")
            else:
                print(f"⚠️ empty node centralities for {combo}")
        except Exception as e:
            print(f"❌ failed node centralities for {combo}: {e}")
    else:
        print(f"⚠️ missing node centralities file for {combo}")

network_metrics_master_df = pd.concat(network_dfs, ignore_index=True) if len(network_dfs) > 0 else pd.DataFrame()
node_centralities_master_df = pd.concat(node_dfs, ignore_index=True) if len(node_dfs) > 0 else pd.DataFrame()

print("\nRebuilt master shapes:")
print("network_metrics_master_df:", network_metrics_master_df.shape)
print("node_centralities_master_df:", node_centralities_master_df.shape)

# overwrite the broken empty master files
network_metrics_master_path = os.path.join(robustness_dir, f"network_metrics_all_combos_{revision}.csv")
node_centralities_master_path = os.path.join(robustness_dir, f"node_centralities_all_combos_{revision}.csv")

network_metrics_master_df.to_csv(network_metrics_master_path, index=False)
node_centralities_master_df.to_csv(node_centralities_master_path, index=False)

print("\n✅ Re-saved repaired master files:")
print(network_metrics_master_path)
print(node_centralities_master_path)

In [ ]:
plot_dir = os.path.join(robustness_dir, "weighted_degree_plots")
os.makedirs(plot_dir, exist_ok=True)

combo_order = [
    "sim_q50_ebc_q50_lc_q25", "sim_q50_ebc_q50_lc_q35", "sim_q50_ebc_q50_lc_q50",
    "sim_q70_ebc_q50_lc_q25", "sim_q70_ebc_q50_lc_q35", "sim_q70_ebc_q50_lc_q50",
    "sim_q85_ebc_q50_lc_q25", "sim_q85_ebc_q50_lc_q35", "sim_q85_ebc_q50_lc_q50",
]

for current_combo in combo_order:
    df_plot = node_centralities_master_df[
        node_centralities_master_df["combo"] == current_combo
    ].copy()

    if df_plot.empty:
        print(f"Skipping {current_combo}: no rows found")
        continue

    tie_types = ["bonding", "bridging_refined", "unclassified"]

    condition_colors = {
        "pre": "seagreen",
        "post": "orangered",
        "counterfactual": "darkred"
    }

    marker_map = {
        "pre": "s",
        "post": "o",
        "counterfactual": "v"
    }

    tie_label_map = {
        "bonding": "Bonding Ties",
        "bridging_refined": "Bridging Ties",
        "unclassified": "Unclassified"
    }

    pre_months = ["Oct2021", "Nov2021"]
    post_months = ["Jan2022", "Feb2022"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)

    for ax, tie_type in zip(axes, tie_types):
        df_tie = df_plot[
            (df_plot["tie_type"] == tie_type) &
            (df_plot["strength"] > 0)
        ].copy()

        if df_tie.empty:
            ax.set_title(tie_label_map[tie_type], fontsize=16)
            ax.axis("off")
            continue

        bins = np.logspace(
            np.log10(df_tie["strength"].min()),
            np.log10(df_tie["strength"].max()),
            25
        )
        bin_centers = np.sqrt(bins[:-1] * bins[1:])

        df_pre = df_plot[
            (df_plot["dataset"] == "observed") &
            (df_plot["phase"] == "pre") &
            (df_plot["tie_type"] == tie_type)
        ].copy()

        pre_curve, pre_mean = average_month_curves(df_pre, pre_months, bins)

        ax.plot(
            bin_centers, pre_curve,
            color=condition_colors["pre"],
            marker=marker_map["pre"],
            markerfacecolor=condition_colors["pre"],
            markeredgecolor=condition_colors["pre"],
            linewidth=2
        )

        if pd.notna(pre_mean) and pre_mean > 0:
            ax.axvline(pre_mean, color=condition_colors["pre"], linestyle="--", linewidth=1.3, alpha=0.9)

        df_post = df_plot[
            (df_plot["dataset"] == "observed") &
            (df_plot["phase"] == "post") &
            (df_plot["tie_type"] == tie_type)
        ].copy()

        post_curve, post_mean = average_month_curves(df_post, post_months, bins)

        ax.plot(
            bin_centers, post_curve,
            color=condition_colors["post"],
            marker=marker_map["post"],
            markerfacecolor=condition_colors["post"],
            markeredgecolor=condition_colors["post"],
            linewidth=2,
            linestyle="-."
        )

        if pd.notna(post_mean) and post_mean > 0:
            ax.axvline(post_mean, color=condition_colors["post"], linestyle=":", linewidth=1.3, alpha=0.9)

        df_rand = df_plot[
            (df_plot["dataset"] == "random") &
            (df_plot["phase"] == "counterfactual") &
            (df_plot["tie_type"] == tie_type)
        ].copy()

        rand_curve, rand_low, rand_high, rand_mean = average_run_curves_with_band(df_rand, bins)

        ax.plot(
            bin_centers, rand_curve,
            color=condition_colors["counterfactual"],
            marker=marker_map["counterfactual"],
            markerfacecolor=condition_colors["counterfactual"],
            markeredgecolor=condition_colors["counterfactual"],
            linewidth=2,
            linestyle="-"
        )

        ax.fill_between(
            bin_centers, rand_low, rand_high,
            color=condition_colors["counterfactual"],
            alpha=0.20
        )

        if pd.notna(rand_mean) and rand_mean > 0:
            ax.axvline(
                rand_mean,
                color=condition_colors["counterfactual"],
                linestyle=(0, (3, 2)),
                linewidth=1.3,
                alpha=0.9
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=16)
        ax.set_title(tie_label_map[tie_type], fontsize=16)

    #axes[0].set_ylabel("P(k)", fontsize=16)
    fig.suptitle(current_combo, fontsize=14)

    plt.subplots_adjust(wspace=0.22)
    plt.tight_layout(rect=[0, 0, 0.88, 0.95])
    plt.show()